In [ ]:
import os
import json
import pickle
import itertools
import numpy as np
from folktables import ACSDataSource

# 1. Downloading the data

We download the ACS dataset using the `folktables` library. 

**Please replace the path in the following cell with the path of the root of this repository**

In [ ]:
repository_path = 'path/to/desia/repository' # Replace this path

In [ ]:
dataset_dir = f'{repository_path}/datasets/acs/'

In [ ]:
all_states = ['AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA', 'HI',
              'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD', 'MA', 'MI',
              'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ', 'NM', 'NY', 'NC',
              'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT',
              'VT', 'VA', 'WA', 'WV', 'WI', 'WY', 'PR']

In [ ]:
data_source = ACSDataSource(survey_year='2018', horizon='1-Year', survey='person', root_dir=os.path.join(dataset_dir, "raw"))

In [ ]:
acs_data = data_source.get_data(states=all_states, download=True)

# 2. Select and save Public Use Microdata Areas (PUMAs)

In [ ]:
puma_sizes = acs_data["PUMA"].value_counts()

In [ ]:
pumas_larger_than_4540 = puma_sizes[puma_sizes > 4540]
acs_smallest_pumas_larger_than_4540 = pumas_larger_than_4540.nsmallest(3)

In [ ]:
acs_smallest_pumas_larger_than_4540

In [ ]:
quasi_ids = ["PUMA", "AGEP", "RAC1P", "SEX"]
seed = 0
np.random.seed(0)

In [ ]:
dataset_dir = f'{repository_path}/datasets/acs-5-year/'
if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)

In [ ]:
for puma_index, num_records in acs_smallest_pumas_larger_than_4540.items():
    puma_raw_data = acs_data[acs_data["PUMA"] == puma_index]
    puma_data = puma_raw_data[quasi_ids]
    # Convert columns to label encoded
    puma_data["PUMA"] = 0
    puma_data["SEX"] -= 1
    puma_data["RAC1P"] -= 1
    # puma_data["LABEL"] = 0
    puma_data["LABEL"] = (puma_raw_data["PINCP"] > 50000).astype(int)
    # Save puma record data
    puma_data.reset_index(drop=True, inplace=True)
    puma_data.to_csv(os.path.join(dataset_dir, "acs_{}.csv".format(puma_index)), index=False)   

# 3. Save the domain of each attribute

In [ ]:
# Save puma domain
domain = {"PUMA": 1, "AGEP": 100, "RAC1P": 9, "SEX": 2, "LABEL": 2}
if not os.path.exists(os.path.join(dataset_dir, "domain")):
    os.makedirs(os.path.join(dataset_dir, "domain"))
for puma_index, _ in acs_smallest_pumas_larger_than_4540.items():
    with open(os.path.join(dataset_dir, "domain", 'acs_{}-domain.json'.format(puma_index)), "w") as fp:
        json.dump(domain, fp)

# 4. Save the queries

In [ ]:
# Save 5-year bucketized puma queries
queries = []
if not os.path.exists(os.path.join(dataset_dir, "queries")):
    os.makedirs(os.path.join(dataset_dir, "queries"))
domain = {"AGEP": 100, "RAC1P": 9, "SEX": 2, "LABEL": 2}
for col in domain:
    bucket = 5 if col == "AGEP" else 1
    for value in range(0, domain[col], bucket):
        one_way_query = {col: np.array(list(range(value, value + bucket))), "PUMA": np.array([0])}
        queries.append(one_way_query)
for col1, col2 in itertools.combinations(domain.keys(), 2):
    bucket1 = 5 if col1 == "AGEP" else 1
    bucket2 = 5 if col2 == "AGEP" else 1
    for value1 in range(0, domain[col1], bucket1):
        for value2 in range(0, domain[col2], bucket2):
            two_way_query = {col1: np.arange(value1, value1 + bucket1), col2: np.arange(value2, value2 + bucket2), "PUMA": np.array([0])}
            queries.append(two_way_query)
for puma_index, _ in acs_smallest_pumas_larger_than_4540.items():
    with open(os.path.join(dataset_dir, "queries", "acs_{}-set.pkl".format(puma_index)), "wb") as fp:
        pickle.dump(queries, fp)